# VisClick — Part C (step 4): assemble source-domain training set

**Prerequisite:** unified Zenodo bundle is on Drive (`data/unified/{train,val,test}/{images,labels}/`). CLAY/VINS images optional (skipped automatically if absent).

**This notebook** (`VisClick_Detailed_Plan.md` §C.5):
1. Build `<DRIVE>/data/source_train/{images,labels}/{train,val,test}/` from the **unified** bundle.
2. **Remap labels** from the 12-class unified taxonomy → the 6 VisClick classes (`button, text, text_input, icon, menu, checkbox`).
3. Write `<DRIVE>/data/source_train/data.yaml` (Ultralytics-ready, absolute Drive path).
4. Patch the repo's `configs/yolo_source.yaml` to point at the same Drive path so §D.1 training uses it directly.
5. Print per-split + per-class **`REPORT`** lines for `VisClick_Report_Data_Form.md` §2.1.

**Idempotent:** each split is skipped if `images/<split>/` already has the target number of files (or set `FORCE = True`).

**Why random (not stratified)?** Reading 84k tiny label files on Drive FUSE is slow and timeout-prone. Random sampling on the unified train pool is a fine v1; the class distribution still follows the population. We log per-class counts so you can decide if a top-up from CLAY/VINS is needed later.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import os, random, shutil, json, time
random.seed(0)

DRIVE   = "/content/drive/MyDrive/visclick"
DATA    = os.path.join(DRIVE, "data")
UNIFIED = os.path.join(DATA, "unified")
OUT     = os.path.join(DATA, "source_train")

# Sample sizes — keeps Colab Free training tractable (see plan §C.5)
N = {"train": 8000, "val": 1000, "test": 1000}
FORCE = False  # set True to rebuild even if source_train/ is populated
MAX_IMG_BYTES = 6_000_000  # skip absurdly large images (val has multi-MB PNGs); 0 disables

CLASSES = ["button", "text", "text_input", "icon", "menu", "checkbox"]

# Unified bundle -> VisClick class mapping (per plan §C.3 table)
UNIFIED_NAMES = ["Button", "Text", "Image", "Icon", "Input", "Link",
                  "Checkbox", "Toggle", "Toolbar", "Navigation", "Modal", "Tab"]
UNIFIED_TO_VIS = {
    "Button": "button", "Toggle": "button", "Tab": "button",
    "Text": "text", "Link": "text",
    "Input": "text_input",
    "Image": "icon", "Icon": "icon",
    "Toolbar": "menu", "Navigation": "menu", "Modal": "menu",
    "Checkbox": "checkbox",
}
UNI_ID_TO_VIS_ID = {
    i: CLASSES.index(UNIFIED_TO_VIS[name]) for i, name in enumerate(UNIFIED_NAMES) if name in UNIFIED_TO_VIS
}
print("unified id -> visclick id =", UNI_ID_TO_VIS_ID)

for sub in ("images", "labels"):
    for sp in ("train", "val", "test"):
        os.makedirs(os.path.join(OUT, sub, sp), exist_ok=True)

print("OUT =", OUT)
print("FORCE =", FORCE, "| MAX_IMG_BYTES =", MAX_IMG_BYTES, "| target sizes =", N)

## 4.1 — Skip-all guard

If `source_train/images/<split>/` already has ≥0.9 × target files for **all three** splits, the assembly is skipped entirely.


In [ ]:
def _count(p):
    try:
        return sum(1 for f in os.listdir(p) if not f.startswith("."))
    except OSError:
        return 0

have = {sp: _count(os.path.join(OUT, "images", sp)) for sp in N}
ok = all(have[sp] >= int(N[sp] * 0.9) for sp in N)
for sp in N:
    print(f"REPORT pre-check | split = {sp:5s} | have = {have[sp]:>5d} | target = {N[sp]:>5d}")
if ok and not FORCE:
    print("REPORT step = ASSEMBLE | status = ALREADY_BUILT | path =", OUT)
    SKIP_ALL = True
else:
    SKIP_ALL = False
    print("REPORT step = ASSEMBLE | status = WILL_BUILD")

## 4.2 — Build each split (random sample + remap)

For each split this cell:
1. Lists the unified `labels/<split>/` directory once (small text files).
2. Shuffles + takes the first `N[split]` entries.
3. For each entry: finds the matching image, **rewrites the label** with mapped class IDs, then copies the image. Images larger than `MAX_IMG_BYTES` are skipped (val contains multi-MB PNGs) and replaced from the next sample.
4. Tracks per-class instance counts.

Skips a split if it already has ≥0.9 × target files (unless `FORCE=True`).


In [ ]:
def _img_for_stem(img_dir, stem):
    for ext in (".jpg", ".jpeg", ".png"):
        p = os.path.join(img_dir, stem + ext)
        if os.path.isfile(p):
            return p
    return None

def _remap_label_text(text):
    out_lines, n_kept = [], 0
    for line in text.splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cid = int(parts[0])
        except ValueError:
            continue
        new_id = UNI_ID_TO_VIS_ID.get(cid)
        if new_id is None:
            continue
        out_lines.append(" ".join([str(new_id)] + parts[1:]))
        n_kept += 1
    return ("\n".join(out_lines) + ("\n" if out_lines else "")), n_kept

summary = {}
if SKIP_ALL:
    print("skipping all splits (already built).")
else:
    for split, target in N.items():
        dst_img = os.path.join(OUT, "images", split)
        dst_lbl = os.path.join(OUT, "labels", split)
        if not FORCE and _count(dst_img) >= int(target * 0.9):
            print(f"skip {split}: already has {_count(dst_img)} images")
            print(f"REPORT step = ASSEMBLE | split = {split} | status = SKIP_BUILT")
            continue

        src_img = os.path.join(UNIFIED, split, "images")
        src_lbl = os.path.join(UNIFIED, split, "labels")
        if not (os.path.isdir(src_img) and os.path.isdir(src_lbl)):
            print(f"REPORT step = ASSEMBLE | split = {split} | status = MISSING_UNIFIED")
            continue

        t0 = time.time()
        all_labels = [f for f in os.listdir(src_lbl) if f.endswith(".txt")]
        random.shuffle(all_labels)
        print(f"{split}: {len(all_labels)} candidate labels in {time.time()-t0:0.1f}s")

        per_class = {c: 0 for c in CLASSES}
        kept = 0
        skipped_big = skipped_empty = skipped_noimg = skipped_err = 0
        i = 0
        # take more than `target` because some are skipped
        while kept < target and i < len(all_labels):
            lname = all_labels[i]; i += 1
            stem = os.path.splitext(lname)[0]
            img_path = _img_for_stem(src_img, stem)
            if img_path is None:
                skipped_noimg += 1
                continue
            try:
                if MAX_IMG_BYTES and os.path.getsize(img_path) > MAX_IMG_BYTES:
                    skipped_big += 1
                    continue
                with open(os.path.join(src_lbl, lname), "r") as fh:
                    raw = fh.read()
            except OSError as e:
                skipped_err += 1
                continue
            new_text, n_kept_lines = _remap_label_text(raw)
            if n_kept_lines == 0:
                skipped_empty += 1
                continue
            try:
                dst_img_path = os.path.join(dst_img, os.path.basename(img_path))
                if not os.path.isfile(dst_img_path):
                    shutil.copy2(img_path, dst_img_path)
                with open(os.path.join(dst_lbl, stem + ".txt"), "w") as fh:
                    fh.write(new_text)
            except OSError as e:
                skipped_err += 1
                continue
            for line in new_text.splitlines():
                cid = int(line.split()[0])
                per_class[CLASSES[cid]] += 1
            kept += 1
            if kept % 500 == 0:
                print(f"  {split}: {kept}/{target} kept ({time.time()-t0:0.0f}s)")

        elapsed = time.time() - t0
        summary[split] = {"kept": kept, "per_class": per_class,
                           "skipped_big": skipped_big, "skipped_empty": skipped_empty,
                           "skipped_noimg": skipped_noimg, "skipped_err": skipped_err,
                           "elapsed_s": round(elapsed, 1)}
        print(f"REPORT step = ASSEMBLE | split = {split} | kept = {kept} | target = {target} | elapsed_s = {elapsed:0.1f}")
        for c in CLASSES:
            print(f"REPORT step = ASSEMBLE | split = {split} | class = {c:11s} | n = {per_class[c]}")
        print(f"REPORT step = ASSEMBLE | split = {split} | skipped_big = {skipped_big} | skipped_empty = {skipped_empty} | skipped_noimg = {skipped_noimg} | skipped_err = {skipped_err}")

## 4.3 — Write `data.yaml` and patch repo config

Writes:
- `<DRIVE>/data/source_train/data.yaml` (absolute Drive path; Ultralytics will read this in step 5).
- `/content/visclick/configs/yolo_source.yaml` patched in-place so the repo copy also points at Drive (no `<DRIVE>` placeholder). The repo file in `git` keeps the placeholder; this patch is **runtime-only** for the current session.


In [ ]:
import yaml as _yaml  # PyYAML is preinstalled in Colab; if not, !pip install pyyaml
data_yaml_path = os.path.join(OUT, "data.yaml")
data_yaml = {
    "path": OUT,
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc": len(CLASSES),
    "names": CLASSES,
}
with open(data_yaml_path, "w") as fh:
    _yaml.safe_dump(data_yaml, fh, sort_keys=False)
print("REPORT step = ASSEMBLE | wrote =", data_yaml_path)
print(open(data_yaml_path).read())

repo_yaml = "/content/visclick/configs/yolo_source.yaml"
if os.path.isfile(repo_yaml):
    with open(repo_yaml, "w") as fh:
        _yaml.safe_dump(data_yaml, fh, sort_keys=False)
    print("REPORT step = ASSEMBLE | patched =", repo_yaml)
else:
    print("NOTE: repo yaml not found (was the repo cloned?):", repo_yaml)

## 4.4 — Final REPORT

Counts in the assembled set + paths. Copy these `REPORT` lines into `VisClick_Report_Data_Form.md` §2.1.


In [ ]:
totals = {c: 0 for c in CLASSES}
for split in ("train", "val", "test"):
    img_n = _count(os.path.join(OUT, "images", split))
    lbl_n = _count(os.path.join(OUT, "labels", split))
    print(f"REPORT final | split = {split:5s} | images = {img_n:>5d} | labels = {lbl_n:>5d}")
    if split in summary:
        for c in CLASSES:
            totals[c] += summary[split]["per_class"][c]
for c in CLASSES:
    print(f"REPORT final | class = {c:11s} | n_instances_total = {totals[c]}")
print("REPORT final | data.yaml =", os.path.join(OUT, "data.yaml"))
print("REPORT final | source_train =", OUT)

**Next:** `notebooks/05_train_source.ipynb` (§D.1) — train YOLOv8s on `source_train/data.yaml` and write weights to `<DRIVE>/weights/baseline_source/...`.

**Optional later top-up** (if a class looks starved in the REPORT counts above):
- Add **CLAY**-mapped images once RICO `combined/` is reachable.
- Add **VINS** images once the external archive is on Drive.
